# How to Diagnose a Disconnected Network
> **Set up**
>
> To run this notebook, first install the Julia kernel for Jupyter Notebooks using [IJulia](https://julialang.github.io/IJulia.jl/stable/manual/installation/), then [create an environment](https://pkgdocs.julialang.org/v1/environments/) for this tutorial with the packages listed with `using <PackageName>` further down.
>
> This tutorial has demonstrated compatibility with these package versions. If you run into any errors, first check your package versions for consistency using `Pkg.status()`.
>
 > ```
 > Status `~/work/PowerNetworkMatrices.jl/PowerNetworkMatrices.jl/docs/Project.toml`
 >   [a93c6f00] DataFrames v1.8.2
 >   [864edb3b] DataStructures v0.19.6
 >   [e30172f5] Documenter v1.19.0
 >   [d12716ef] DocumenterInterLinks v1.1.0
 >   [98b081ad] Literate v2.21.0
 >   [bed98974] PowerNetworkMatrices v0.24.4 `~/work/PowerNetworkMatrices.jl/PowerNetworkMatrices.jl`
 >   [f00506e0] PowerSystemCaseBuilder v2.6.0
 >   [bcd98974] PowerSystems v5.12.3
 >   [08abe8d2] PrettyTables v3.4.8
 > 
 > ```



A singular `ABA` matrix or a failed DC power flow is frequently just a
disconnected network: an island with no reference bus leaves `ABA` singular.
Checking connectivity first localizes the problem before you dig into the numerics.
This guide walks through the check, then deliberately breaks a network so you can see
exactly what a fragmented result looks like — and how to get back.

In [ ]:
using PowerNetworkMatrices
import PowerSystems
import PowerSystemCaseBuilder

Load an example test system with `PowerSystemCaseBuilder.build_system`:

In [ ]:
sys = PowerSystemCaseBuilder.build_system(
    PowerSystemCaseBuilder.PSITestSystems,
    "c_sys5",
);

## Step 1 — Confirm a healthy network is connected

`validate_connectivity` returns `true` when the system forms a single
connected component:

In [ ]:
validate_connectivity(sys)

`find_subnetworks` shows the decomposition behind that answer: a `Dict`
mapping each island's reference bus to the set of bus numbers in it. A connected
system yields a **single** entry:

In [ ]:
find_subnetworks(sys)

`validate_connectivity` and `find_subnetworks` also accept an
already-built `AdjacencyMatrix` or `Ybus`, so a matrix you already
have on hand is reused instead of rebuilt:

In [ ]:
adj = AdjacencyMatrix(sys)
validate_connectivity(adj)

## Step 2 — Disconnect a bus and watch it split

To see a fragmented result on a real system, let's isolate one bus. A bus goes silent
when every branch touching it is out of service, so we find bus `5`'s incident
branches and mark them unavailable with
`PowerSystems.set_available!` — `Ybus` (and therefore the connectivity check)
only includes available branches:

In [ ]:
isolated_bus = 5
incident = [
    br for br in PowerSystems.get_components(PowerSystems.ACBranch, sys) if
    PowerSystems.get_number(PowerSystems.get_from(PowerSystems.get_arc(br))) ==
    isolated_bus ||
    PowerSystems.get_number(PowerSystems.get_to(PowerSystems.get_arc(br))) == isolated_bus
]

for br in incident
    PowerSystems.set_available!(br, false)
end

The network is now split. `validate_connectivity` reports it:

In [ ]:
validate_connectivity(sys)

...and `find_subnetworks` returns **two** entries — the main island, and bus `5`
stranded on its own:

In [ ]:
find_subnetworks(sys)

There is the diagnosis. That second island — the isolated `{5}` — has no reference
bus of its own, which is exactly the block that would have made `ABA` singular. The
bus set tells you precisely which buses to reconnect (or which island to study in
isolation). Here it points straight back at the bus we broke.

## Step 3 — Reconnect and recover

Restoring the branches we took out puts the network back together — bus `5` rejoins
the main island and `validate_connectivity` is `true` again:

In [ ]:
for br in incident
    PowerSystems.set_available!(br, true)
end

validate_connectivity(sys)

...and `find_subnetworks` is back to a single island, identical to where we
started:

In [ ]:
find_subnetworks(sys)

## Choosing a traversal algorithm

The lower-level `find_subnetworks(M, bus_numbers; subnetwork_algorithm)` — which runs
over a raw sparse connectivity matrix — lets you pick how the graph is walked:

  - `iterative_union_find` (the **default**) — an iterative union-find
    disjoint-set, safe on networks of any size.
  - `depth_first_search` — a recursive traversal.

Both return the **same** island decomposition, so the choice is about performance,
not correctness. Prefer the default union-find; it avoids the deep recursion that
`depth_first_search` can hit on very large networks. The `subnetwork_algorithm`
keyword also threads through the matrix constructors, so islands are detected the same
way at build time as by an explicit `find_subnetworks` call — here an
`ABA_Matrix` built with `depth_first_search`:

In [ ]:
ABA_Matrix(sys; subnetwork_algorithm = depth_first_search);

## See also

  - Matrix overview & indexing — the `AdjacencyMatrix` and
    `Ybus` graphs these checks traverse, and per-island axes.
  - Network Reduction Theory — how the susceptance graph can fragment into
    more islands than the admittance graph, and why that matters for `ABA`.